# 01 — Activation patching

Notebook 00 confirmed the behavior: GPT-2 small completes the IOI prompt with the indirect object, logit diff ≈ **+3.2**. This notebook answers the first *where* question: **which (layer, position) activations causally carry that behavior?**

The tool is **activation patching** (README has the long version):

1. Run a **clean** prompt — behavior present.
2. Run a **corrupted** prompt — one token changed, behavior flipped.
3. Re-run the corrupted prompt, but at *one* chosen internal location, splice in the activation from the clean run. If the behavior comes back, that location was carrying the signal.

Sweep the splice point over every layer × position and you get a map of where the circuit lives.

**This notebook is exercises, not a demo.** Each exercise has a `TODO` cell, a checkpoint with the expected numbers (verified on this exact model + prompts), and a collapsed solution below it. The intended loop:

> **predict → write code → run → compare against your prediction**

Keep the log below honest — the gap between expected and observed is the curriculum.

| # | What I predicted | What happened |
|---|------------------|---------------|
| 2 | *(fill in before running)* | |
| 3 | | |
| 4 | | |
| 5 | | |


In [ ]:
import torch
from transformer_lens import HookedTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)          # interventions only, no training

model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()
print(f"device: {device}  layers={model.cfg.n_layers}  heads={model.cfg.n_heads}")

## The clean / corrupted pair

Corrupting well is half the technique. We change exactly **one token** — the second occurrence of the subject (`S2`) — so the indirect object flips:

- **clean**: "… the store, **John** gave a drink to" → answer " Mary"
- **corrupted**: "… the store, **Mary** gave a drink to" → answer " John"

Two properties make this corruption good:

1. **Token alignment is preserved** — both prompts tokenize to the same length, so position *i* means the same thing in both runs. Patching across misaligned prompts is meaningless.
2. **The answer flips rather than degrades** — the logit diff swings from ≈ +3.2 to ≈ −3.5, a wide, signed range. Patching one location either pulls the metric back toward clean or it doesn't; there's no ambiguity about what "restored" means.

In [ ]:
clean_prompt     = "When John and Mary went to the store, John gave a drink to"
corrupted_prompt = "When John and Mary went to the store, Mary gave a drink to"

io_tok = model.to_single_token(" Mary")   # correct answer on the CLEAN prompt
s_tok  = model.to_single_token(" John")

clean_tokens     = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)
assert clean_tokens.shape == corrupted_tokens.shape, "corruption broke token alignment"

str_toks = model.to_str_tokens(clean_prompt)
n_pos = len(str_toks)
print(f"{n_pos} positions:")
print(list(enumerate(str_toks)))

S2, END = 10, n_pos - 1   # the swapped token's position, and the final position

## Exercise 1 — baselines and the patching metric

Compute the logit diff on both prompts, then define the **normalized metric** every patch will be scored with:

$$\text{metric}(\text{logits}) = \frac{\text{logit\_diff} - \text{corrupted\_ld}}{\text{clean\_ld} - \text{corrupted\_ld}}$$

So **0 = fully corrupted, 1 = fully restored**, and a patch is read as "what fraction of the behavior did this single activation bring back." (It can go below 0 or above 1 — a patch can *hurt*, or overshoot. Both happen later. Worth thinking now about what each would mean.)

You'll also want both runs cached — `run_with_cache` — since every patch needs the clean activations to splice in.

**Checkpoint:** `clean_ld ≈ +3.172`, `corrupted_ld ≈ −3.535`, `metric(clean) = 1.0`, `metric(corrupted) = 0.0`.

In [ ]:
def logit_diff(logits) -> float:
    """IO minus S logit at the final position. logits: [batch, pos, d_vocab]"""
    # TODO
    ...

# TODO: run both prompts with run_with_cache, keep logits + caches
clean_logits, clean_cache = ...
corrupted_logits, corrupted_cache = ...

CLEAN_LD = logit_diff(clean_logits)
CORR_LD  = logit_diff(corrupted_logits)

def ioi_metric(logits) -> float:
    """0 = corrupted baseline, 1 = clean baseline."""
    # TODO
    ...

print(f"clean logit_diff:     {CLEAN_LD:+.3f}   (expect ≈ +3.172)")
print(f"corrupted logit_diff: {CORR_LD:+.3f}   (expect ≈ −3.535)")
print(f"metric(clean)     = {ioi_metric(clean_logits):.3f}   (expect 1.000)")
print(f"metric(corrupted) = {ioi_metric(corrupted_logits):.3f}   (expect 0.000)")

<details><summary><b>Solution — Exercise 1</b></summary>

```python
def logit_diff(logits) -> float:
    return (logits[0, -1, io_tok] - logits[0, -1, s_tok]).item()

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

CLEAN_LD = logit_diff(clean_logits)
CORR_LD  = logit_diff(corrupted_logits)

def ioi_metric(logits) -> float:
    return (logit_diff(logits) - CORR_LD) / (CLEAN_LD - CORR_LD)
```

The metric is a linear rescale of the logit diff, nothing deeper — but normalizing matters because "+1.3 logit diff" is uninterpretable on its own, while "0.72 of the behavior restored" reads directly.
</details>

## A heatmap with no plotting deps

The repo deliberately has no matplotlib; a colored HTML table is plenty for 12×15 grids and keeps the environment lean. Cells show **metric × 100** ("% of behavior restored"): red = restored, blue = made it worse, white = no effect.

In [ ]:
from IPython.display import HTML, display

def heatmap(grid, col_labels, row_labels, title=""):
    """grid: anything indexable as [row][col] of metric values (≈ −0.3 … 1.05)."""
    def color(v):
        x = max(min(float(v), 1.05), -0.35)
        if x >= 0:
            t = x / 1.05
            return f"rgb(255,{int(255 * (1 - 0.75 * t))},{int(255 * (1 - 0.85 * t))})"
        t = x / -0.35
        return f"rgb({int(255 * (1 - 0.75 * t))},{int(255 * (1 - 0.5 * t))},255)"
    th = "padding:2px 7px;font-size:11px"
    head = "<tr><th></th>" + "".join(f"<th style='{th}'>{c}</th>" for c in col_labels) + "</tr>"
    body = "".join(
        f"<tr><th style='{th};text-align:right'>{rl}</th>" + "".join(
            f"<td style='background:{color(v)};{th};text-align:right'>{float(v)*100:.0f}</td>"
            for v in row) + "</tr>"
        for rl, row in zip(row_labels, grid))
    display(HTML(f"<div style='font-family:monospace'><b>{title}</b>"
                 f"<table style='border-collapse:collapse'>{head}{body}</table></div>"))

heatmap([[0.0, 0.5, 1.0], [-0.3, 0.0, 0.3]], ["a", "b", "c"], ["r0", "r1"], "render test")

## Exercise 2 — patch the residual stream by (layer, position)

The main event. For every layer L and position P: run the corrupted prompt, and at hook `blocks.{L}.hook_resid_pre` overwrite position P with the clean run's value. Score with `ioi_metric`. That's 12 × 15 = 180 forward passes (seconds on GPU, a couple of minutes on CPU).

The splice is a hook function — same shape as the README example:

```python
def hook(act, hook):                  # act: [batch, pos, d_model]
    act[:, P, :] = clean_cache["resid_pre", L][:, P, :]
    return act
model.run_with_hooks(corrupted_tokens, fwd_hooks=[(f"blocks.{L}.hook_resid_pre", hook)])
```

**Predict before you run** (write it in the log up top). The one token that differs sits at position 10 (`S2`). The answer is read out at position 14 (`END`). So: at layer 0, which positions can possibly matter? At layer 11, where must the signal be? And in between — does it move gradually, or jump?

**Checkpoint:** row L0 should be ≈ 100 at `S2` and ≈ 0 everywhere else. Row L11 should be ≈ 100 at `END`. The hand-off happens around **L8–L9** (at L8: S2 ≈ 57, END ≈ 22; at L9: S2 ≈ 14, END ≈ 87).

In [ ]:
resid = torch.zeros(model.cfg.n_layers, n_pos)

# TODO: fill resid[layer, pos] with ioi_metric of the patched run.
# Mind the closure trap: bind layer/pos per iteration (default args or a factory),
# or every hook will see the loop's final values.
...

heatmap(resid, str_toks, [f"L{i}" for i in range(model.cfg.n_layers)],
        "resid_pre patching — % of behavior restored")

<details><summary><b>Solution — Exercise 2</b></summary>

```python
def patch_resid(layer, pos):
    def hook(act, hook):
        act[:, pos, :] = clean_cache["resid_pre", layer][:, pos, :]
        return act
    logits = model.run_with_hooks(
        corrupted_tokens, fwd_hooks=[(f"blocks.{layer}.hook_resid_pre", hook)])
    return ioi_metric(logits)

for layer in range(model.cfg.n_layers):
    for pos in range(n_pos):
        resid[layer, pos] = patch_resid(layer, pos)
```

The factory function (`patch_resid` returning a fresh `hook`) is what binds `layer`/`pos` correctly. The classic bug is defining the hook inline in the loop body and referencing the loop variables — every hook then patches the *last* (layer, pos).
</details>

### Reading the map

Two columns light up and nothing else:

- **L0–L7, column `S2`** — early on, the entire causal story is "which name sits at position 10." Patch that one residual vector and the corrupted run becomes the clean run, because everything downstream just recomputes from it.
- **L9–L11, column `END`** — by the late layers the information has been *moved*: the final position's residual stream now contains "the answer is Mary," and patching it there restores ≈ 100% even though position 10 still holds the corrupted name.
- **The hand-off is L8–L9** — S2's restorative power collapses (57 → 14) exactly as END's rises (22 → 87). Some set of attention heads in those layers reads from S2-ish positions and writes to END. *Which* heads is the next notebook's question.

Note what the map does **not** show: nothing restores at the other name positions, `gave`, `drink`… The circuit is narrow — two positions, one corridor between them.

## Exercise 3 — attention or MLP?

Same sweep, two more hook points: `blocks.{L}.hook_attn_out` and `blocks.{L}.hook_mlp_out` — the attention block's total write to the residual stream, and the MLP's. This decomposes the resid map: *who* is doing the writing at each layer?

**Predict first.** Moving information *between positions* is the one thing only attention can do (MLPs act per-position). So which of the two grids should light up at `END` around L7–9? And could an MLP still matter at `S2`?

**Checkpoint:** `attn_out` peaks at END: L7 ≈ 21, **L8 ≈ 40**, L9 ≈ 16 — and L11/END goes *negative* (≈ −21). `mlp_out` is ≈ 0 everywhere except **L0/S2 ≈ 77**.

In [ ]:
grids = {}
for name in ("attn_out", "mlp_out"):
    grid = torch.zeros(model.cfg.n_layers, n_pos)
    # TODO: same sweep as exercise 2, but patching f"blocks.{layer}.hook_{name}"
    ...
    grids[name] = grid
    heatmap(grid, str_toks, [f"L{i}" for i in range(model.cfg.n_layers)],
            f"{name} patching — % of behavior restored")

<details><summary><b>Solution — Exercise 3</b></summary>

```python
def patch_block(layer, pos, name):
    def hook(act, hook):
        act[:, pos, :] = clean_cache[name, layer][:, pos, :]
        return act
    logits = model.run_with_hooks(
        corrupted_tokens, fwd_hooks=[(f"blocks.{layer}.hook_{name}", hook)])
    return ioi_metric(logits)

for name in ("attn_out", "mlp_out"):
    grid = torch.zeros(model.cfg.n_layers, n_pos)
    for layer in range(model.cfg.n_layers):
        for pos in range(n_pos):
            grid[layer, pos] = patch_block(layer, pos, name)
    grids[name] = grid
```
</details>

### Reading the two maps

- **Attention at END, L7–L9, is the corridor** — those are the heads moving "which name was duplicated" into the final position. Notice no single layer restores everything (40% at best): the work is *distributed* across several layers' heads, which add their contributions into the residual stream.
- **L11 attn_out at END is negative** ≈ −21: patching it in actively *hurts*. There are heads that push against the IOI answer (you'll meet a famous one in the next exercise). A patch below zero is your first evidence that circuits contain opposing forces, not just helpers.
- **MLPs barely matter — except L0 at S2 (≈ 77)**. This is a known GPT-2 quirk: layer-0's MLP acts as an "extended embedding" that the model uses to compute what a token *is*; for duplicate-name detection that identity signal is most of the story at S2. The honest takeaway isn't "MLPs do nothing," it's "this particular *moving* circuit is attention-shaped, with one embedding-flavored MLP exception."

Why these grids don't sum to the resid map: resid_pre patching replaces the *accumulated* stream (everything upstream at that position), while attn/mlp patching replaces one layer's *increment*. Increments are small and redundant; the accumulation is total.

## Exercise 4 — which heads? (preview of notebook 02)

`attn_out` is the sum of 12 heads' outputs. Patch each head's output (`z`) individually — hook `blocks.{L}.attn.hook_z`, shape `[batch, pos, head, d_head]` — across **all positions at once**, giving a 12 × 12 (layer, head) grid. 144 runs.

**Predict:** given the layer story so far, where should the big heads sit? How big can one head be, if all of L8's attention combined was 40%?

**Checkpoint — top 5 heads:** `8.10 ≈ 19.5`, `8.6 ≈ 15.2`, `9.9 ≈ 12.1`, `7.3 ≈ 11.9`, `5.5 ≈ 11.5`. And one head is wildly *negative*: `10.7 ≈ −29.5`.

In [ ]:
headgrid = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

# TODO: per (layer, head), patch clean_cache["z", layer][:, :, head, :] into the
# corrupted run at hook f"blocks.{layer}.attn.hook_z" — all positions at once.
...

heatmap(headgrid, [f"H{h}" for h in range(model.cfg.n_heads)],
        [f"L{i}" for i in range(model.cfg.n_layers)],
        "per-head z patching — % of behavior restored")

flat = headgrid.flatten()
top = flat.abs().topk(6).indices
print("biggest heads (|metric|):")
for i in top:
    print(f"  L{i.item() // model.cfg.n_heads}.H{i.item() % model.cfg.n_heads}"
          f"  {flat[i].item() * 100:+.1f}")

<details><summary><b>Solution — Exercise 4</b></summary>

```python
def patch_head(layer, head):
    def hook(act, hook):                       # act: [batch, pos, head, d_head]
        act[:, :, head, :] = clean_cache["z", layer][:, :, head, :]
        return act
    logits = model.run_with_hooks(
        corrupted_tokens, fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook)])
    return ioi_metric(logits)

for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        headgrid[layer, head] = patch_head(layer, head)
```
</details>

### The cast list

The handful of heads that light up are exactly the families Wang et al. name — this grid is the IOI paper in miniature:

| heads | metric | family (per the paper) |
|-------|--------|------------------------|
| 7.3, 8.6, 8.10 | +12…+20 | **S-inhibition** — write "suppress the duplicated subject" into END |
| 9.9 (and 9.6, 10.0) | +12 | **name movers** — copy the remaining name into the output |
| 5.5 (and 6.9) | +11 | **induction / duplicate-token** — detect that a name repeated |
| 10.7 (and 11.10) | **−29** | **negative name movers** — actively push against the answer |

Head 10.7 is the famous one: it consistently writes *against* the correct name. Why a trained model contains heads that fight its own answer is still an open research question (calibration? backup? training dynamics artifact?) — but you just found it yourself with 144 forward passes.

Notebook 02 takes each family apart: what each head attends to, what it writes, and how they compose.

## Sidebar — patching ≠ ablation (falsify yourself)

Patching said head 9.9 carries +12% of the behavior. A natural stronger claim: "9.9 is *necessary* — kill it and IOI breaks."

**Predict what `logit_diff` does on the clean prompt if 9.9's output is zeroed.** Then run it.

In [ ]:
def ablate_99(act, hook):
    act[:, :, 9, :] = 0.0
    return act

abl_logits = model.run_with_hooks(clean_tokens,
                                  fwd_hooks=[("blocks.9.attn.hook_z", ablate_99)])
print(f"clean logit_diff, head 9.9 zeroed: {logit_diff(abl_logits):+.3f}"
      f"   (was {CLEAN_LD:+.3f})")

It barely moves: **+3.17 → +3.10**. The strong claim is falsified — and the reason is a real phenomenon: **backup name movers**. When 9.9 is knocked out, other heads (10.0, 9.6, and the backup family in L10–11) shift their behavior and pick up the slack. The circuit is redundant, hydra-style.

Two lessons worth keeping:

1. **Patching and ablation answer different questions.** Patching asks "does this activation carry the clean/corrupted *difference*?" Ablation asks "is this component *necessary* given everything else still running?" A head can carry real signal and still not be necessary.
2. **Always try to break your own circuit story.** If you'd stopped at the patching grid, "9.9 is critical" would have felt proven. One falsification run later, the claim needs the word "redundantly."

---

**Where things stand:** the IOI signal lives at `S2` until L7, crosses to `END` through attention in L7–9, and is written by a small named cast of heads — duplicate detectors, S-inhibition, name movers, and their negative counterparts. **Next: `02_head_attribution.ipynb`** — what each head actually attends to and writes, i.e. *how* the corridor works.